# HealthBot — Streamlit + LangGraph (packaged from a notebook)

This notebook writes out the Streamlit app's source files (`state.py`, `nodes.py`, `graph.py`, `app.py`) to disk, then launches the app for you.

**Important:** Streamlit apps don't render *inside* a notebook cell — they run as their own web server that opens in your browser. Run the cells top to bottom; the last cell starts the server and gives you a local URL to open.

Uses **Groq** (`langchain-groq`, model `openai/gpt-oss-120b`) and **Tavily** for search.

## 1. Install dependencies (run once)

In [1]:
# !pip install streamlit langgraph langchain-core langchain-groq langchain-tavily python-dotenv


## 2. Set your API keys

Create a `config.env` file next to this notebook with:
```
GROQ_API_KEY="gsk_your_groq_api_key_here"
TAVILY_API_KEY="tvly-your_tavily_key_here"
```
Get a Groq key: https://console.groq.com/keys · Get a Tavily key: https://app.tavily.com/home

In [2]:
config_env = '''GROQ_API_KEY="gsk_your_groq_api_key_here"
TAVILY_API_KEY="tvly-your_tavily_key_here"
'''

import os
if not os.path.exists("config.env"):
    with open("config.env", "w") as f:
        f.write(config_env)
    print("Created config.env — open it and paste in your real API keys before continuing.")
else:
    print("config.env already exists — leaving it as is.")


config.env already exists — leaving it as is.


## 3. Write the app's source files

Each cell below uses `%%writefile` to save one module to disk, exactly as they'd exist in a normal project folder.

In [3]:
%%writefile state.py
"""
State shared across all nodes in the HealthBot workflow.
Every node reads/writes to this; LangGraph merges partial
updates back into it automatically.
"""

from typing import TypedDict, Optional


class HealthBotState(TypedDict, total=False):
    topic: str
    search_results: str
    summary: str
    quiz_question: str
    quiz_answer: str
    grade: str
    explanation: str
    want_new_topic: Optional[bool]


Overwriting state.py


In [4]:
%%writefile nodes.py
"""
Every step of the HealthBot flow is its own node, each with a single
responsibility, exactly per the rubric.

Nodes that need a human response use LangGraph's `interrupt()`. Calling
interrupt(payload) pauses graph execution and returns `payload` to the
caller (Streamlit) as part of an `__interrupt__` entry in the result.
Execution resumes exactly where it paused when the caller invokes the
graph again with `Command(resume=<value>)` - the interrupt() call then
returns <value> instead of pausing again.
"""

from langgraph.types import interrupt
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage, SystemMessage

from state import HealthBotState

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)
tavily = TavilySearch(max_results=5, topic="general")


# ---------------------------------------------------------------------
# 1. Ask the patient what topic they want to learn about
# ---------------------------------------------------------------------
def get_topic_node(state: HealthBotState) -> dict:
    topic = interrupt({"prompt": "What health topic or condition would you like to learn about?"})
    return {"topic": topic}


# ---------------------------------------------------------------------
# 2. Search Tavily for reputable medical info
# ---------------------------------------------------------------------
def search_node(state: HealthBotState) -> dict:
    query = f"{state['topic']} patient education reputable medical source"
    results = tavily.invoke({"query": query})
    # results["results"] is a list of {title, url, content, ...}
    formatted = "\n\n".join(
        f"Source: {r['title']} ({r['url']})\n{r['content']}"
        for r in results.get("results", [])
    )
    return {"search_results": formatted}


# ---------------------------------------------------------------------
# 3. Summarize into patient-friendly language
# ---------------------------------------------------------------------
SUMMARY_PROMPT = """You are a patient education assistant. Using ONLY the \
information below (no outside knowledge), write a clear, patient-friendly \
3-4 paragraph explanation of "{topic}". Avoid jargon; explain any medical \
term you must use. Do not add facts that aren't in the source material.

SOURCE MATERIAL:
{search_results}
"""


def summarize_node(state: HealthBotState) -> dict:
    prompt = SUMMARY_PROMPT.format(topic=state["topic"], search_results=state["search_results"])
    response = llm.invoke([SystemMessage(content=prompt)])
    return {"summary": response.content}


# ---------------------------------------------------------------------
# 4. Let the patient read the summary, wait until they're ready
# ---------------------------------------------------------------------
def comprehension_ready_node(state: HealthBotState) -> dict:
    interrupt({"prompt": "Take your time reading the summary above. Click 'Ready' when you'd like a comprehension check.", "summary": state["summary"]})
    return {}


# ---------------------------------------------------------------------
# 5. Generate one quiz question from the summary ONLY
# ---------------------------------------------------------------------
QUIZ_PROMPT = """Based ONLY on the following patient-friendly summary, write \
ONE quiz question that tests the patient's understanding. The question must \
be answerable using only the summary. Return only the question text, no \
preamble, no answer.

SUMMARY:
{summary}
"""


def quiz_node(state: HealthBotState) -> dict:
    response = llm.invoke([SystemMessage(content=QUIZ_PROMPT.format(summary=state["summary"]))])
    return {"quiz_question": response.content.strip()}


# ---------------------------------------------------------------------
# 6. Present the question, wait for the patient's answer
# ---------------------------------------------------------------------
def get_answer_node(state: HealthBotState) -> dict:
    answer = interrupt({"prompt": "Answer the question:", "question": state["quiz_question"]})
    return {"quiz_answer": answer}


# ---------------------------------------------------------------------
# 7. Grade the answer using ONLY the summary, with citations
# ---------------------------------------------------------------------
GRADE_PROMPT = """You are grading a patient's answer to a comprehension question. \
Use ONLY the summary below as your source of truth - do not use outside \
medical knowledge.

SUMMARY:
{summary}

QUESTION:
{question}

PATIENT'S ANSWER:
{answer}

Respond in this exact format:
Grade: <A/B/C/D/F>
Explanation: <1-2 sentences on why, quoting or closely referencing the \
relevant part of the summary that supports or corrects the answer>
"""


def grade_node(state: HealthBotState) -> dict:
    prompt = GRADE_PROMPT.format(
        summary=state["summary"],
        question=state["quiz_question"],
        answer=state["quiz_answer"],
    )
    response = llm.invoke([SystemMessage(content=prompt)]).content.strip()

    grade, explanation = "N/A", response
    if "Grade:" in response and "Explanation:" in response:
        grade_part, explanation_part = response.split("Explanation:", 1)
        grade = grade_part.replace("Grade:", "").strip()
        explanation = explanation_part.strip()

    return {"grade": grade, "explanation": explanation}


# ---------------------------------------------------------------------
# 8. Present the grade, ask: new topic or exit?
# ---------------------------------------------------------------------
def continue_node(state: HealthBotState) -> dict:
    choice = interrupt({
        "prompt": "Would you like to learn about another topic, or exit?",
        "grade": state["grade"],
        "explanation": state["explanation"],
    })
    return {"want_new_topic": choice == "new"}


# ---------------------------------------------------------------------
# 9. Reset state before looping back, for privacy + accuracy
# ---------------------------------------------------------------------
def reset_node(state: HealthBotState) -> dict:
    return {
        "topic": "",
        "search_results": "",
        "summary": "",
        "quiz_question": "",
        "quiz_answer": "",
        "grade": "",
        "explanation": "",
        "want_new_topic": None,
    }


Overwriting nodes.py


In [5]:
%%writefile graph.py
"""
Wires the nodes into a LangGraph StateGraph with a checkpointer.
A checkpointer is REQUIRED for interrupt()/Command(resume=...) to work -
it's what lets the graph pause and be resumed later (even across
Streamlit reruns) instead of losing its place.
"""

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

from state import HealthBotState
from nodes import (
    get_topic_node,
    search_node,
    summarize_node,
    comprehension_ready_node,
    quiz_node,
    get_answer_node,
    grade_node,
    continue_node,
    reset_node,
)


def _route_after_continue(state: HealthBotState) -> str:
    return "reset" if state.get("want_new_topic") else "end"


def build_graph():
    graph = StateGraph(HealthBotState)

    graph.add_node("get_topic", get_topic_node)
    graph.add_node("search", search_node)
    graph.add_node("summarize", summarize_node)
    graph.add_node("comprehension_ready", comprehension_ready_node)
    graph.add_node("quiz", quiz_node)
    graph.add_node("get_answer", get_answer_node)
    graph.add_node("grade", grade_node)
    graph.add_node("continue_check", continue_node)
    graph.add_node("reset", reset_node)

    graph.add_edge(START, "get_topic")
    graph.add_edge("get_topic", "search")
    graph.add_edge("search", "summarize")
    graph.add_edge("summarize", "comprehension_ready")
    graph.add_edge("comprehension_ready", "quiz")
    graph.add_edge("quiz", "get_answer")
    graph.add_edge("get_answer", "grade")
    graph.add_edge("grade", "continue_check")
    graph.add_conditional_edges(
        "continue_check",
        _route_after_continue,
        {"reset": "reset", "end": END},
    )
    graph.add_edge("reset", "get_topic")

    checkpointer = MemorySaver()
    return graph.compile(checkpointer=checkpointer)


Overwriting graph.py


In [6]:
%%writefile app.py
"""
Streamlit front-end. Each rerun either:
  (a) starts the graph for the first time, or
  (b) resumes it from its last interrupt() with Command(resume=...)

The graph's own state (topic, summary, etc.) lives in LangGraph's
checkpointer, keyed by thread_id - NOT in st.session_state. Streamlit
only needs to remember: the thread_id, and the latest interrupt payload
so it knows what to render.
"""

import uuid
import streamlit as st
from dotenv import load_dotenv
from langgraph.types import Command

from graph import build_graph

load_dotenv("config.env")

st.set_page_config(page_title="HealthBot", page_icon="🩺")
st.title("🩺 HealthBot — Patient Education Assistant")

if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())
    st.session_state.graph = build_graph()
    st.session_state.interrupt = None
    st.session_state.done = False

config = {"configurable": {"thread_id": st.session_state.thread_id}}
graph = st.session_state.graph


def advance(resume_value=None):
    """Run the graph until it either finishes or hits the next interrupt()."""
    if resume_value is None:
        result = graph.invoke({}, config)
    else:
        result = graph.invoke(Command(resume=resume_value), config)

    if "__interrupt__" in result:
        st.session_state.interrupt = result["__interrupt__"][0].value
    else:
        st.session_state.interrupt = None
        st.session_state.done = True


# Kick off the graph on first load
if st.session_state.interrupt is None and not st.session_state.done:
    advance()

payload = st.session_state.interrupt

# ---------------------------------------------------------------------
# Render whichever step the graph is currently paused at
# ---------------------------------------------------------------------
if st.session_state.done:
    st.success("Session ended. Refresh the page to start over.")

elif payload and "question" not in payload and "grade" not in payload and "summary" not in payload:
    # get_topic step
    st.write(payload["prompt"])
    topic = st.text_input("Topic")
    if st.button("Submit", disabled=not topic):
        advance(topic)
        st.rerun()

elif payload and "summary" in payload:
    # comprehension_ready step
    st.subheader(f"About: {payload.get('summary', '')[:0]}")  # no-op placeholder for clarity
    st.markdown(payload["summary"])
    st.info(payload["prompt"])
    if st.button("I'm ready for the quiz"):
        advance("ready")
        st.rerun()

elif payload and "question" in payload:
    # get_answer step
    st.markdown(f"**Quiz question:** {payload['question']}")
    answer = st.text_area("Your answer")
    if st.button("Submit answer", disabled=not answer):
        advance(answer)
        st.rerun()

elif payload and "grade" in payload:
    # continue_check step
    st.markdown(f"**Grade:** {payload['grade']}")
    st.markdown(f"**Explanation:** {payload['explanation']}")
    col1, col2 = st.columns(2)
    if col1.button("Learn about another topic"):
        advance("new")
        st.rerun()
    if col2.button("Exit"):
        advance("exit")
        st.rerun()


Overwriting app.py


## 4. Launch the Streamlit app

This starts Streamlit as a background process and prints the local URL to open in your browser. Leave this cell's process running while you use the app; re-run the cell above (or restart the kernel) if you edit the source files.

In [7]:
import sys
!{sys.executable} -m pip install streamlit

In [8]:
import subprocess, sys, time

streamlit_process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.headless", "true", "--server.port", "8001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(5)
if streamlit_process.poll() is not None:
    out, err = streamlit_process.communicate()
    print("Process exited early!")
    print("STDOUT:", out.decode(errors="ignore"))
    print("STDERR:", err.decode(errors="ignore"))
else:
    print("Still running — open http://localhost:8001")


Still running — open http://localhost:8001


### Stop the app

Run this cell when you're done, so the background server doesn't keep running after you close the notebook.

In [9]:
# streamlit_process.terminate()
# print("Streamlit stopped.")


In [10]:
import os, requests
from dotenv import load_dotenv
load_dotenv("config.env")

key = os.getenv("TAVILY_API_KEY")
print("Key loaded:", key[:8] + "..." if key else "MISSING")

resp = requests.post(
    "https://api.tavily.com/search",
    json={"api_key": key, "query": "cancer patient education reputable medical source", "max_results": 3},
)
print("Status code:", resp.status_code)
print(resp.json())

Key loaded: tvly-dev...
Status code: 200
{'query': 'cancer patient education reputable medical source', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.patientpower.info/navigating-cancer/understanding-cancer/one-letter-makes-a-difference-avoiding-online-research-pitfalls', 'title': 'How to Find Reliable Cancer Information Online', 'content': 'Dr. Tubb encourages patients to start with established, reputable sources and organizations, such as the American Cancer Society (ACS), the American Society of Clinical Oncology (ASCO), or the National Cancer Institute (NCI). Non-profit health care organizations and hospital systems can also be reliable sources of accurate health information. [...] National Comprehensive Cancer Network (NCCN): This is an excellent resource for evidence-based treatment guidelines. The website also offers patient education materials.\n National Cancer Institute-Designated Cancer Centers: These are NCI-funded treatment fac